# AutoQual AI+ — Web Attack Classifier (CSIC + ECML)

Trains **Model 2 — Web Attack Classification** on the merged CSIC 2010 / ECML web-attack
dataset. Input is a raw HTTP request (method + headers + body/query), output is
`Valid` vs `Anomalous`.

**Pipeline:** load → EDA → feature engineering → preprocessing pipeline → train/test split
→ baseline model comparison → hyperparameter tuning → final evaluation → save artifact
(`.pkl` + metadata) for deployment behind the FastAPI inference service.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


In [ ]:
# Core imports for this notebook
import re
import json
import warnings
import importlib.metadata as importlib_metadata

import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    accuracy_score, precision_recall_fscore_support
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
RANDOM_STATE = 42


## 1. Load Dataset

In [ ]:
DATA_PATH = '/kaggle/input/datasets/hloworld1/csic-ecml/csic_ecml_normalized_final.csv'

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()


## 2. Exploratory Data Analysis

In [ ]:
df.info()


In [ ]:
print('Missing values per column:')
print(df.isnull().sum().sort_values(ascending=False))


In [ ]:
print(df['Class'].value_counts())
print()
print(df['Class'].value_counts(normalize=True).round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Class'].value_counts().plot(kind='bar', ax=axes[0], color=['#4c9a2a', '#c0392b'])
axes[0].set_title('Class balance')
axes[0].set_xlabel('')

df['Method'].value_counts().plot(kind='bar', ax=axes[1], color='#2a6f9a')
axes[1].set_title('HTTP method distribution')
axes[1].set_xlabel('')
plt.tight_layout()
plt.show()


In [ ]:
# A quick look at what an anomalous vs. a valid request actually looks like
pd.set_option('display.max_colwidth', 120)
print('--- Sample ANOMALOUS request ---')
print(df[df['Class'] == 'Anomalous'][['Method', 'POST-Data', 'GET-Query']].dropna(how='all', subset=['POST-Data', 'GET-Query']).sample(2, random_state=RANDOM_STATE))
print()
print('--- Sample VALID request ---')
print(df[df['Class'] == 'Valid'][['Method', 'POST-Data', 'GET-Query']].dropna(how='all', subset=['POST-Data', 'GET-Query']).sample(2, random_state=RANDOM_STATE))


## 3. Feature Engineering

Two families of signal, combined:
- **Text** — every free-text field (headers, body, query string) concatenated into one
  blob and TF-IDF vectorized, so attack tokens (`<script`, `union select`, `../`, etc.)
  show up as features regardless of which field they land in.
- **Engineered numeric/categorical** — lengths, special-character counts, a suspicious-keyword
  regex hit count, and whether the request even has a body/query. These are cheap, robust
  signals that boost a plain TF-IDF baseline noticeably on this kind of dataset.


In [ ]:
SUSPICIOUS_PATTERNS = [
    r"select\s+.*\s+from", r"union\s+select", r"drop\s+table", r"insert\s+into",
    r"<script", r"javascript:", r"onerror\s*=", r"onload\s*=",
    r"\.\./", r"etc/passwd", r"cmd\.exe", r"/bin/sh",
    r"%3cscript", r"%27", r"--\s", r";\s*--", r"exec\s*\(", r"alert\s*\("
]
SUSPICIOUS_RE = re.compile("|".join(SUSPICIOUS_PATTERNS), re.IGNORECASE)

TEXT_COLS = [
    'Host-Header', 'Accept', 'Accept-Charset', 'Accept-Language',
    'Cache-control', 'Pragma', 'User-Agent', 'Content-Type',
    'POST-Data', 'GET-Query'
]

def engineer_features(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Fills missing text fields and derives the numeric/categorical signals used by the model."""
    out = raw_df.copy()

    for col in TEXT_COLS:
        out[col] = out[col].fillna('')

    out['combined_text'] = out[TEXT_COLS].agg(' '.join, axis=1)

    out['post_len'] = out['POST-Data'].str.len()
    out['query_len'] = out['GET-Query'].str.len()
    out['combined_len'] = out['combined_text'].str.len()
    out['special_char_count'] = out['combined_text'].apply(
        lambda t: sum(t.count(c) for c in ['<', '>', "'", '"', ';', '%'])
    )
    out['digit_count'] = out['combined_text'].apply(lambda t: sum(ch.isdigit() for ch in t))
    out['suspicious_keyword_count'] = out['combined_text'].apply(lambda t: len(SUSPICIOUS_RE.findall(t)))
    out['has_post_data'] = (out['POST-Data'] != '').astype(int)
    out['has_get_query'] = (out['GET-Query'] != '').astype(int)

    out['Method'] = out['Method'].fillna('UNKNOWN')
    out['Connection'] = out['Connection'].fillna('unknown')

    return out

df_feat = engineer_features(df)
df_feat[['post_len', 'query_len', 'combined_len', 'special_char_count',
         'digit_count', 'suspicious_keyword_count']].describe()


In [ ]:
# Sanity check: engineered features should clearly separate the two classes
df_feat.groupby('Class')[['suspicious_keyword_count', 'special_char_count', 'combined_len']].mean()


## 4. Preprocessing Pipeline

In [ ]:
NUMERIC_FEATURES = [
    'post_len', 'query_len', 'combined_len', 'special_char_count',
    'digit_count', 'suspicious_keyword_count', 'has_post_data', 'has_get_query'
]
CATEGORICAL_FEATURES = ['Method', 'Connection']
TEXT_FEATURE = 'combined_text'

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUMERIC_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
    ('text', TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2), TEXT_FEATURE),
])


## 5. Train / Test Split

In [ ]:
y = (df_feat['Class'] == 'Anomalous').astype(int)  # 1 = Anomalous (attack), 0 = Valid
LABEL_MAP = {0: 'Valid', 1: 'Anomalous'}

X_train, X_test, y_train, y_test = train_test_split(
    df_feat, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print('Train:', X_train.shape, ' Test:', X_test.shape)
print('Train class balance:', y_train.value_counts(normalize=True).round(3).to_dict())


## 6. Baseline Model Comparison

In [ ]:
candidate_models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(
        n_estimators=200, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
    ),
}

results = []
fitted_pipelines = {}

for name, clf in candidate_models.items():
    pipe = Pipeline([('preprocess', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    preds = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary')

    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc_score(y_test, proba),
    })

    print(f'=== {name} ===')
    print(classification_report(y_test, preds, target_names=['Valid', 'Anomalous']))

results_df = pd.DataFrame(results).sort_values('roc_auc', ascending=False).reset_index(drop=True)
results_df


In [ ]:
results_df.set_index('model')[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']].plot(
    kind='bar', figsize=(10, 5), ylim=(0, 1)
)
plt.title('Baseline model comparison')
plt.ylabel('score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

BEST_MODEL_NAME = results_df.iloc[0]['model']
print('Best baseline model by ROC-AUC:', BEST_MODEL_NAME)


## 7. Hyperparameter Tuning

Tunes whichever model won the baseline comparison above, using `RandomizedSearchCV`
(faster than a full grid search on ~85k rows) optimizing for ROC-AUC.


In [ ]:
PARAM_GRIDS = {
    'LogisticRegression': {
        'clf__C': [0.01, 0.1, 1, 10, 100],
        'clf__penalty': ['l2'],
    },
    'RandomForest': {
        'clf__n_estimators': [200, 400, 600],
        'clf__max_depth': [None, 10, 20, 40],
        'clf__min_samples_split': [2, 5, 10],
        'clf__min_samples_leaf': [1, 2, 4],
    },
    'XGBoost': {
        'clf__n_estimators': [200, 400, 600],
        'clf__max_depth': [3, 5, 7, 9],
        'clf__learning_rate': [0.01, 0.05, 0.1, 0.2],
        'clf__subsample': [0.7, 0.85, 1.0],
        'clf__colsample_bytree': [0.7, 0.85, 1.0],
    },
}

def make_tuning_estimator(model_name):
    # Fresh instance with n_jobs pinned to 1 — RandomizedSearchCV below already
    # parallelizes across folds/candidates with n_jobs=-1; also letting the model
    # itself spawn n_jobs=-1 workers nests two parallel pools inside each other,
    # which is wasteful at best and a known source of flaky/silent failures
    # (deadlocks, NaN CV scores) on some platforms at worst.
    if model_name == 'LogisticRegression':
        return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    if model_name == 'RandomForest':
        return RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)
    return XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=1)

base_pipeline = Pipeline([('preprocess', preprocessor), ('clf', make_tuning_estimator(BEST_MODEL_NAME))])

search = RandomizedSearchCV(
    base_pipeline,
    param_distributions=PARAM_GRIDS[BEST_MODEL_NAME],
    n_iter=15,
    scoring='roc_auc',
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)

print('Best CV ROC-AUC:', search.best_score_)
print('Best params:', search.best_params_)

best_pipeline = search.best_estimator_


## 8. Final Evaluation (tuned model, held-out test set)

In [ ]:
final_preds = best_pipeline.predict(X_test)
final_proba = best_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, final_preds, target_names=['Valid', 'Anomalous']))
final_roc_auc = roc_auc_score(y_test, final_proba)
print('Final ROC-AUC:', final_roc_auc)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, final_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Valid', 'Anomalous'], yticklabels=['Valid', 'Anomalous'])
axes[0].set_title('Confusion matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

fpr, tpr, _ = roc_curve(y_test, final_proba)
axes[1].plot(fpr, tpr, label=f'ROC-AUC = {final_roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[1].set_title('ROC curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()


## 9. Save Model Artifact for Deployment

Saves the **full pipeline** (preprocessing + tuned model bundled together) so inference-time
feature engineering always matches training-time exactly — plus a metadata file describing
the input schema, label mapping, and library versions needed to reproduce this environment
in the FastAPI serving service.


In [ ]:
MODEL_OUT_PATH = '/kaggle/working/csic_web_attack_model.pkl'
METADATA_OUT_PATH = '/kaggle/working/model_metadata.json'

joblib.dump(best_pipeline, MODEL_OUT_PATH)

def get_version(pkg):
    try:
        return importlib_metadata.version(pkg)
    except importlib_metadata.PackageNotFoundError:
        return None

metadata = {
    'model_name': 'csic_ecml_web_attack_classifier',
    'best_model_type': BEST_MODEL_NAME,
    'best_params': search.best_params_,
    'label_map': LABEL_MAP,
    'positive_class': 'Anomalous',
    'raw_input_columns': [
        'Method', 'Host-Header', 'Connection', 'Accept', 'Accept-Charset',
        'Accept-Language', 'Cache-control', 'Pragma', 'User-Agent',
        'Content-Type', 'POST-Data', 'GET-Query'
    ],
    'note': 'Pass a DataFrame with the raw_input_columns above through engineer_features() '
            'before calling pipeline.predict() / predict_proba() — the pipeline itself only '
            'handles the ColumnTransformer stage (numeric/categorical/text), not the '
            'engineer_features() feature-derivation step.',
    'test_metrics': {
        'accuracy': accuracy_score(y_test, final_preds),
        'roc_auc': final_roc_auc,
    },
    'library_versions': {
        'python': __import__('sys').version,
        'scikit-learn': get_version('scikit-learn'),
        'pandas': get_version('pandas'),
        'numpy': get_version('numpy'),
        'xgboost': get_version('xgboost'),
        'joblib': get_version('joblib'),
    },
}

with open(METADATA_OUT_PATH, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print('Saved model to:', MODEL_OUT_PATH)
print('Saved metadata to:', METADATA_OUT_PATH)
print()
print(json.dumps(metadata, indent=2, default=str))


### Handoff checklist
When downloading the output of this notebook, grab both:
- `csic_web_attack_model.pkl` — the trained pipeline
- `model_metadata.json` — schema, label mapping, params, and versions

The `engineer_features()` function above also needs to travel with the model — the pipeline
only covers the `ColumnTransformer` stage, not the raw-request → engineered-columns step.
